In [63]:
import sys
import os
import pandas as pd

from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config.settings import RAW_DATA_DIR, INTERIM_DATA_DIR

In [64]:
data = os.path.join(INTERIM_DATA_DIR, "precos_comb_unificados.csv")
df = pd.read_csv(data)

C:\Users\Luiz\AppData\Local\Temp\ipykernel_12652\574276380.py:2: DtypeWarning: Columns (0: CNPJ da Revenda) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data)


In [65]:
print(df.isna().sum())

Regiao - Sigla          9114
Estado - Sigla          9114
Municipio               9114
Revenda                 9114
CNPJ da Revenda         9114
Nome da Rua             9114
Numero Rua             10312
Complemento          3373470
Bairro                 17824
Cep                     9114
Produto                 9114
Data da Coleta          9114
Valor de Venda          9114
Valor de Compra      4331306
Unidade de Medida       9114
Bandeira                9114
dtype: int64


In [66]:
df.shape

(4331306, 16)

In [67]:
df.columns

Index(['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda',
       'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro',
       'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra',
       'Unidade de Medida', 'Bandeira'],
      dtype='str')

1º Avaliar a importância de cada coluna para futuras análise
2º Formatar o nome da coluna para todos os caracteres minúsculos e sem espaço 

In [68]:
df.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA,01/01/2021,"4,599",NaN,R$ / litro,BRANCA
1,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,ETANOL,01/01/2021,"4,199",NaN,R$ / litro,BRANCA
2,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA ADITIVADA,01/01/2021,"4,799",NaN,R$ / litro,BRANCA
3,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL,01/01/2021,"3,499",NaN,R$ / litro,BRANCA
4,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL S10,01/01/2021,"3,599",NaN,R$ / litro,BRANCA


Região - Sigla = regiao
Estado Silga = estado
Revenda = posto
CNPJ da Revenda = Remover!
Nome da Rua ?
Numero da rua = remover
complemento = remover
Bairro = bairro
CEP = Remover
Produto = produto
Data de Coleta = data_coleta
Valor de Venda = preco
Valor de Compra = Remover
Unidade de Medida = unidade_medida
Bandeira = bandeira

In [69]:
df.columns

Index(['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda',
       'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro',
       'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra',
       'Unidade de Medida', 'Bandeira'],
      dtype='str')

In [70]:
df = df.drop(columns=['CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Cep', 'Valor de Compra', 'Complemento'])
df.columns

Index(['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'Bairro',
       'Produto', 'Data da Coleta', 'Valor de Venda', 'Unidade de Medida',
       'Bandeira'],
      dtype='str')

In [71]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '', regex=False)
    .str.replace('__', '_')
)

In [72]:
df.columns

Index(['regiao_sigla', 'estado_sigla', 'municipio', 'revenda', 'bairro',
       'produto', 'data_da_coleta', 'valor_de_venda', 'unidade_de_medida',
       'bandeira'],
      dtype='str')

In [73]:
df.rename(columns={'regiao_sigla': 'regiao'}, inplace=True)
df.rename(columns={'estado_sigla': 'estado'}, inplace=True)
df.rename(columns={'revenda': 'posto'}, inplace=True)
df.columns

Index(['regiao', 'estado', 'municipio', 'posto', 'bairro', 'produto',
       'data_da_coleta', 'valor_de_venda', 'unidade_de_medida', 'bandeira'],
      dtype='str')

Verificação de dados Nan ou Null

há muitos registros totalmente em branco, por se tratar de um registro sem qualuqer dado, vamos remove-los por completo

In [74]:
df = df.dropna(how='all')
print(df.isna().sum())

regiao                  0
estado                  0
municipio               0
posto                   0
bairro               8710
produto                 0
data_da_coleta          0
valor_de_venda          0
unidade_de_medida       0
bandeira                0
dtype: int64


Temos também, muitos valores vazios na coluna bairro, para não alterar a essencia da df, atribuiremos o valor desconhecido registros de bairro que forem Nan

In [75]:
df['bairro'] = df['bairro'].fillna('desconhecido')
print(df.isna().sum())

regiao               0
estado               0
municipio            0
posto                0
bairro               0
produto              0
data_da_coleta       0
valor_de_venda       0
unidade_de_medida    0
bandeira             0
dtype: int64


In [76]:
print(df.isnull().sum())

regiao               0
estado               0
municipio            0
posto                0
bairro               0
produto              0
data_da_coleta       0
valor_de_venda       0
unidade_de_medida    0
bandeira             0
dtype: int64


Verificou-se qu não há valores null

In [80]:
for coluna in df.columns:
    print(f'{coluna} : {df[coluna].nunique()}')

regiao : 5
estado : 27
municipio : 465
posto : 17140
bairro : 7124
produto : 6
data_da_coleta : 1345
valor_de_venda : 4091
unidade_de_medida : 3
bandeira : 73


In [81]:
df['unidade_de_medida'].unique()

<StringArray>
['R$ / litro', 'R$ / m³', 'R$ / m3']
Length: 3, dtype: str